In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

her_dir = '/content/drive/MyDrive/BAH_hackathon_data'
for f in os.listdir(her_dir):
    size = os.path.getsize(os.path.join(her_dir, f)) / 1e6
    print(f"{f}: {size:.1f} MB")

Mounted at /content/drive
goes_14year_raw_PROGRESS.csv: 353.0 MB
goes_14year_raw.csv: 335.1 MB
flares_cache.csv: 0.6 MB
goes_preprocessed.csv: 1159.0 MB


In [ ]:
import pandas as pd
import os

her_dir = '/content/drive/MyDrive/BAH_hackathon_data'

# Check her preprocessed file
goes = pd.read_csv(os.path.join(her_dir, 'goes_preprocessed.csv'), nrows=5)
print("Columns:", goes.columns.tolist())
print("Sample:\n", goes.head())

# Check row count
row_count = sum(1 for _ in open(os.path.join(her_dir, 'goes_preprocessed.csv'))) - 1
print(f"\nTotal rows: {row_count:,}")

# Check raw file
raw = pd.read_csv(os.path.join(her_dir, 'goes_14year_raw.csv'), nrows=5)
print("\nRaw columns:", raw.columns.tolist())
raw_count = sum(1 for _ in open(os.path.join(her_dir, 'goes_14year_raw.csv'))) - 1
print(f"Raw rows: {raw_count:,}")

Columns: ['time', 'segment_id', 'log_soft', 'log_hard', 'log_hardening', 'dFs_dt', 'dFh_dt', 'Fs_ma5', 'd2Fs_dt2']
Sample:
                   time  segment_id  log_soft  log_hard  log_hardening  \
0  2010-04-07 19:45:00           0 -6.527897 -7.067048      -0.539151   
1  2010-04-07 19:46:00           0 -6.599683 -7.147447      -0.547764   
2  2010-04-07 19:47:00           0 -6.677568 -7.251760      -0.574192   
3  2010-04-07 19:48:00           0 -6.733735 -7.339090      -0.605355   
4  2010-04-07 19:49:00           0 -6.776567 -7.429275      -0.652708   

     dFs_dt    dFh_dt    Fs_ma5  d2Fs_dt2  
0  0.000000  0.000000 -6.601716  0.000000  
1 -0.071786 -0.080399 -6.634721 -0.071786  
2 -0.077886 -0.104314 -6.663090 -0.006100  
3 -0.056167 -0.087330 -6.717603  0.021719  
4 -0.042832 -0.090185 -6.761228  0.013335  

Total rows: 7,318,564

Raw columns: ['time', 'hard', 'soft']
Raw rows: 7,318,573


In [ ]:
import pandas as pd
import numpy as np
import os
import gc

HER_DIR  = '/content/drive/MyDrive/BAH_hackathon_data'
MY_DIR   = '/content/drive/MyDrive/isro_hackathon_data'

FEATURE_COLS = ['log_soft', 'log_hard', 'log_hardening',
                'dFs_dt', 'dFh_dt', 'Fs_ma5', 'd2Fs_dt2']
WINDOW     = 60
HORIZON    = 30
N_FEATURES = 7

# ── 1. LOAD HER PREPROCESSED DATA ────────────────────────────────────────────
print("Loading her preprocessed data...")
goes = pd.read_csv(os.path.join(HER_DIR, 'goes_preprocessed.csv'))
goes['time'] = pd.to_datetime(goes['time'], format='mixed')
goes = goes[goes['time'].dt.year <= 2023].reset_index(drop=True)
print(f"Rows: {len(goes):,} | Segments: {goes['segment_id'].nunique()}")

# ── 2. LOAD FLARE CATALOG FROM YOUR DRIVE ────────────────────────────────────
flares   = pd.read_csv(os.path.join(MY_DIR, 'flares_cache.csv'), parse_dates=['start_time'])
mx_times = np.sort(flares[flares['class'].str[0].isin(['M', 'X'])]['start_time'].values)
print(f"M/X flares: {len(mx_times)}")

# ── 3. COUNT TOTAL WINDOWS ───────────────────────────────────────────────────
print("Counting total windows...")
total_windows = 0
for seg_id, seg_df in goes.groupby('segment_id'):
    n = len(seg_df)
    if n >= WINDOW + HORIZON:
        total_windows += n - WINDOW - HORIZON
print(f"Total windows: {total_windows:,}")

# ── 4. ALLOCATE MEMMAPS ───────────────────────────────────────────────────────
X_path = os.path.join(MY_DIR, 'X_windows_full.npy')
y_path = os.path.join(MY_DIR, 'y_labels_full.npy')
t_path = os.path.join(MY_DIR, 'window_times_full.npy')

X_mm = np.lib.format.open_memmap(X_path, mode='w+', dtype='float32',
                                   shape=(total_windows, WINDOW, N_FEATURES))
y_mm = np.lib.format.open_memmap(y_path, mode='w+', dtype='int8',
                                   shape=(total_windows,))
t_mm = np.lib.format.open_memmap(t_path, mode='w+', dtype='int64',
                                   shape=(total_windows,))

# ── 5. WINDOWING ─────────────────────────────────────────────────────────────
def has_flare(window_end, horizon_end, mx_times):
    return np.searchsorted(mx_times, horizon_end, side='right') > \
           np.searchsorted(mx_times, window_end,  side='left')

idx    = 0
segs   = list(goes.groupby('segment_id'))
n_segs = len(segs)

for seg_num, (seg_id, seg_df) in enumerate(segs):
    seg_df = seg_df.reset_index(drop=True)
    n = len(seg_df)
    if n < WINDOW + HORIZON:
        continue
    values    = seg_df[FEATURE_COLS].values.astype(np.float32)
    times_arr = seg_df['time'].values
    for i in range(n - WINDOW - HORIZON):
        window_end  = times_arr[i + WINDOW]
        horizon_end = window_end + np.timedelta64(HORIZON, 'm')
        X_mm[idx]   = values[i:i+WINDOW]
        y_mm[idx]   = 1 if has_flare(window_end, horizon_end, mx_times) else 0
        t_mm[idx]   = window_end.astype('int64')
        idx += 1
    if seg_num % 100 == 0:
        X_mm.flush(); y_mm.flush(); t_mm.flush()
        print(f"  Segment {seg_num}/{n_segs} | windows: {idx:,}")

X_mm.flush(); y_mm.flush(); t_mm.flush()
print(f"\nDone. Total written: {idx:,}")
print(f"Positive rate: {y_mm[:idx].mean()*100:.4f}%")
print(f"Positive count: {y_mm[:idx].sum():,}")

del goes
gc.collect()

Loading her preprocessed data...
Rows: 6,803,568 | Segments: 2010
M/X flares: 1311
Counting total windows...
Total windows: 6,628,735
  Segment 0/2010 | windows: 495
  Segment 100/2010 | windows: 215,372
  Segment 200/2010 | windows: 525,615
  Segment 300/2010 | windows: 775,545
  Segment 400/2010 | windows: 1,053,414
  Segment 500/2010 | windows: 1,274,173
  Segment 600/2010 | windows: 1,534,181
  Segment 700/2010 | windows: 1,770,077
  Segment 800/2010 | windows: 2,029,952
  Segment 900/2010 | windows: 2,259,899
  Segment 1000/2010 | windows: 2,495,458
  Segment 1100/2010 | windows: 2,734,141
  Segment 1200/2010 | windows: 3,092,362
  Segment 1300/2010 | windows: 3,931,840
  Segment 1400/2010 | windows: 4,215,355
  Segment 1600/2010 | windows: 4,984,180
  Segment 1700/2010 | windows: 5,460,054
  Segment 1800/2010 | windows: 5,751,837
  Segment 1900/2010 | windows: 6,217,961
  Segment 2000/2010 | windows: 6,517,851

Done. Total written: 6,628,735
Positive rate: 0.5778%
Positive count:

0

In [ ]:
import numpy as np
import pandas as pd
import os
import gc
from sklearn.preprocessing import StandardScaler
import joblib

MY_DIR     = '/content/drive/MyDrive/isro_hackathon_data'
N_FEATURES = 7
WINDOW     = 60

# ── 1. LOAD MEMMAPS ──────────────────────────────────────────────────────────
print("Loading memmaps...")
X = np.load(os.path.join(MY_DIR, 'X_windows_full.npy'), mmap_mode='r')
y = np.load(os.path.join(MY_DIR, 'y_labels_full.npy'),  mmap_mode='r')
t = np.load(os.path.join(MY_DIR, 'window_times_full.npy'), mmap_mode='r')
print(f"X: {X.shape} | positives: {y.sum():,}")

times = pd.to_datetime(t, unit='ns')
years = times.year

# ── 2. YEAR-BASED SPLIT ───────────────────────────────────────────────────────
train_idx = np.where((years >= 2010) & (years <= 2020))[0]
val_idx   = np.where((years >= 2021) & (years <= 2022))[0]
test_idx  = np.where(years == 2023)[0]

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

print(f"\nTrain: {len(train_idx):,} | positives: {y_train.sum():,}")
print(f"Val:   {len(val_idx):,} | positives: {y_val.sum():,}")
print(f"Test:  {len(test_idx):,} | positives: {y_test.sum():,}")

# ── 3. FIT SCALER ON TRAIN SAMPLE ────────────────────────────────────────────
print("\nFitting scaler...")
sample_idx = train_idx[np.random.choice(len(train_idx), size=300_000, replace=False)]
scaler = StandardScaler()
scaler.fit(X[sample_idx].reshape(-1, N_FEATURES))
del sample_idx
gc.collect()

# ── 4. SCALE AND SAVE ────────────────────────────────────────────────────────
def scale_and_save(X_full, indices, out_path, scaler, n_features, window):
    total = len(indices)
    out   = np.lib.format.open_memmap(
        out_path, mode='w+', dtype='float32',
        shape=(total, window, n_features)
    )
    CHUNK = 10_000
    for start in range(0, total, CHUNK):
        end        = min(start + CHUNK, total)
        chunk      = X_full[indices[start:end]]
        out[start:end] = scaler.transform(
            chunk.reshape(-1, n_features)
        ).reshape(chunk.shape).astype('float32')
    out.flush()
    print(f"  Saved {os.path.basename(out_path)} — {total:,} windows")

print("Scaling and saving splits...")
scale_and_save(X, train_idx, os.path.join(MY_DIR, 'X_train_full.npy'), scaler, N_FEATURES, WINDOW)
scale_and_save(X, val_idx,   os.path.join(MY_DIR, 'X_val_full.npy'),   scaler, N_FEATURES, WINDOW)
scale_and_save(X, test_idx,  os.path.join(MY_DIR, 'X_test_full.npy'),  scaler, N_FEATURES, WINDOW)

np.save(os.path.join(MY_DIR, 'y_train_full.npy'), y_train)
np.save(os.path.join(MY_DIR, 'y_val_full.npy'),   y_val)
np.save(os.path.join(MY_DIR, 'y_test_full.npy'),  y_test)

joblib.dump(scaler, os.path.join(MY_DIR, 'standard_scaler_full.pkl'))

print("\n✅ All full splits saved.")
print(f"Train: {y_train.sum():,} / {len(y_train):,} ({y_train.mean()*100:.4f}%)")
print(f"Val:   {y_val.sum():,} / {len(y_val):,} ({y_val.mean()*100:.4f}%)")
print(f"Test:  {y_test.sum():,} / {len(y_test):,} ({y_test.mean()*100:.4f}%)")

Loading memmaps...
X: (6628735, 60, 7) | positives: 38,303

Train: 5,112,853 | positives: 22,397
Val:   1,009,455 | positives: 6,390
Test:  506,427 | positives: 9,516

Fitting scaler...
Scaling and saving splits...
  Saved X_train_full.npy — 5,112,853 windows
  Saved X_val_full.npy — 1,009,455 windows
  Saved X_test_full.npy — 506,427 windows

✅ All full splits saved.
Train: 22,397 / 5,112,853 (0.4381%)
Val:   6,390 / 1,009,455 (0.6330%)
Test:  9,516 / 506,427 (1.8790%)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

MY_DIR = '/content/drive/MyDrive/isro_hackathon_data'

files = {
    # Original partial data splits
    'X_train_goes.npy':       'Train windows (partial data)',
    'X_val_goes.npy':         'Val windows (partial data)',
    'X_test_goes.npy':        'Test windows (partial data)',
    'y_train_goes.npy':       'Train labels (partial data)',
    'y_val_goes.npy':         'Val labels (partial data)',
    'y_test_goes.npy':        'Test labels (partial data)',
    'standard_scaler.pkl':    'Scaler (partial data)',
    # Full data splits
    'X_train_full.npy':       'Train windows (FULL data)',
    'X_val_full.npy':         'Val windows (FULL data)',
    'X_test_full.npy':        'Test windows (FULL data)',
    'y_train_full.npy':       'Train labels (FULL data)',
    'y_val_full.npy':         'Val labels (FULL data)',
    'y_test_full.npy':        'Test labels (FULL data)',
    'standard_scaler_full.pkl': 'Scaler (FULL data)',
    # Raw and preprocessed
    'goes_14year_raw_PROGRESS.csv': 'Raw GOES data',
    'goes_preprocessed.csv':        'Preprocessed (partial)',
    'flares_cache.csv':             'Flare catalog',
    # Window checkpoints
    'X_windows_raw.npy':      'Raw windows (partial)',
    'X_windows_full.npy':     'Raw windows (FULL)',
    'y_labels_raw.npy':       'Raw labels (partial)',
    'y_labels_full.npy':      'Raw labels (FULL)',
    'window_times_raw.npy':   'Raw times (partial)',
    'window_times_full.npy':  'Raw times (FULL)',
}

print(f"{'File':<35} {'Description':<30} {'Size':>10}  Status")
print("-" * 85)
for fname, desc in files.items():
    path = os.path.join(MY_DIR, fname)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f"{fname:<35} {desc:<30} {size:>8.1f}MB  ✅")
    else:
        print(f"{fname:<35} {desc:<30} {'':>10}  ❌ MISSING")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File                                Description                          Size  Status
-------------------------------------------------------------------------------------
X_train_goes.npy                    Train windows (partial data)     8059.4MB  ✅
X_val_goes.npy                      Val windows (partial data)        957.5MB  ✅
X_test_goes.npy                     Test windows (partial data)       470.5MB  ✅
y_train_goes.npy                    Train labels (partial data)         4.8MB  ✅
y_val_goes.npy                      Val labels (partial data)           0.6MB  ✅
y_test_goes.npy                     Test labels (partial data)          0.3MB  ✅
standard_scaler.pkl                 Scaler (partial data)               0.0MB  ✅
X_train_full.npy                    Train windows (FULL data)        8589.6MB  ✅
X_val_full.npy                      Val windows (FU

In [ ]:
import os

MY_DIR = '/content/drive/MyDrive/isro_hackathon_data'

rename_map = {
    # Partial files — add _partial suffix
    'X_train_goes.npy':           'X_train_partial.npy',
    'X_val_goes.npy':             'X_val_partial.npy',
    'X_test_goes.npy':            'X_test_partial.npy',
    'y_train_goes.npy':           'y_train_partial.npy',
    'y_val_goes.npy':             'y_val_partial.npy',
    'y_test_goes.npy':            'y_test_partial.npy',
    'standard_scaler.pkl':        'standard_scaler_partial.pkl',
    'X_windows_raw.npy':          'X_windows_partial.npy',
    'y_labels_raw.npy':           'y_labels_partial.npy',
    'window_times_raw.npy':       'window_times_partial.npy',
    'goes_preprocessed.csv':      'goes_preprocessed_partial.csv',
    'goes_14year_raw_PROGRESS.csv': 'goes_14year_raw_partial.csv',
    # Full files — clean names
    'X_train_full.npy':           'X_train.npy',
    'X_val_full.npy':             'X_val.npy',
    'X_test_full.npy':            'X_test.npy',
    'y_train_full.npy':           'y_train.npy',
    'y_val_full.npy':             'y_val.npy',
    'y_test_full.npy':            'y_test.npy',
    'standard_scaler_full.pkl':   'standard_scaler.pkl',
    'X_windows_full.npy':         'X_windows.npy',
    'y_labels_full.npy':          'y_labels.npy',
    'window_times_full.npy':      'window_times.npy',
}

for old_name, new_name in rename_map.items():
    old_path = os.path.join(MY_DIR, old_name)
    new_path = os.path.join(MY_DIR, new_name)
    if os.path.exists(old_path):
        os.rename(old_path, new_path)
        print(f"✅ {old_name} → {new_name}")
    else:
        print(f"❌ Not found: {old_name}")

print("\nFinal file list:")
for f in sorted(os.listdir(MY_DIR)):
    size = os.path.getsize(os.path.join(MY_DIR, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

✅ X_train_goes.npy → X_train_partial.npy
✅ X_val_goes.npy → X_val_partial.npy
✅ X_test_goes.npy → X_test_partial.npy
✅ y_train_goes.npy → y_train_partial.npy
✅ y_val_goes.npy → y_val_partial.npy
✅ y_test_goes.npy → y_test_partial.npy
✅ standard_scaler.pkl → standard_scaler_partial.pkl
✅ X_windows_raw.npy → X_windows_partial.npy
✅ y_labels_raw.npy → y_labels_partial.npy
✅ window_times_raw.npy → window_times_partial.npy
✅ goes_preprocessed.csv → goes_preprocessed_partial.csv
✅ goes_14year_raw_PROGRESS.csv → goes_14year_raw_partial.csv
✅ X_train_full.npy → X_train.npy
✅ X_val_full.npy → X_val.npy
✅ X_test_full.npy → X_test.npy
✅ y_train_full.npy → y_train.npy
✅ y_val_full.npy → y_val.npy
✅ y_test_full.npy → y_test.npy
✅ standard_scaler_full.pkl → standard_scaler.pkl
✅ X_windows_full.npy → X_windows.npy
✅ y_labels_full.npy → y_labels.npy
✅ window_times_full.npy → window_times.npy

Final file list:
  X_test.npy: 850.8 MB
  X_test_partial.npy: 470.5 MB
  X_train.npy: 8589.6 MB
  X_train_part

In [ ]:
import os
import shutil

MY_DIR      = '/content/drive/MyDrive/isro_hackathon_data'
PARTIAL_DIR = '/content/drive/MyDrive/isro_hackathon_data_partial'

os.makedirs(PARTIAL_DIR, exist_ok=True)

partial_files = [
    'X_train_partial.npy',
    'X_val_partial.npy',
    'X_test_partial.npy',
    'y_train_partial.npy',
    'y_val_partial.npy',
    'y_test_partial.npy',
    'standard_scaler_partial.pkl',
    'X_windows_partial.npy',
    'y_labels_partial.npy',
    'window_times_partial.npy',
    'goes_preprocessed_partial.csv',
    'goes_14year_raw_partial.csv',
]

for fname in partial_files:
    src = os.path.join(MY_DIR, fname)
    dst = os.path.join(PARTIAL_DIR, fname)
    if os.path.exists(src):
        shutil.move(src, dst)
        print(f"✅ Moved: {fname}")
    else:
        print(f"❌ Not found: {fname}")

print("\nMain folder now contains:")
for f in sorted(os.listdir(MY_DIR)):
    size = os.path.getsize(os.path.join(MY_DIR, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

✅ Moved: X_train_partial.npy
✅ Moved: X_val_partial.npy
✅ Moved: X_test_partial.npy
✅ Moved: y_train_partial.npy
✅ Moved: y_val_partial.npy
✅ Moved: y_test_partial.npy
✅ Moved: standard_scaler_partial.pkl
✅ Moved: X_windows_partial.npy
✅ Moved: y_labels_partial.npy
✅ Moved: window_times_partial.npy
✅ Moved: goes_preprocessed_partial.csv
✅ Moved: goes_14year_raw_partial.csv

Main folder now contains:
  X_test.npy: 850.8 MB
  X_train.npy: 8589.6 MB
  X_val.npy: 1695.9 MB
  X_windows.npy: 11136.3 MB
  flares_cache.csv: 0.5 MB
  goes_14year_raw.csv: 265.6 MB
  standard_scaler.pkl: 0.0 MB
  window_times.npy: 53.0 MB
  y_labels.npy: 6.6 MB
  y_test.npy: 0.5 MB
  y_train.npy: 5.1 MB
  y_val.npy: 1.0 MB


In [ ]:
import shutil
import os

HER_DIR = '/content/drive/MyDrive/BAH_hackathon_data'
MY_DIR  = '/content/drive/MyDrive/isro_hackathon_data'

print("Copying full preprocessed file... (this may take a minute, it's 1.1GB)")
shutil.copy(
    os.path.join(HER_DIR, 'goes_preprocessed.csv'),
    os.path.join(MY_DIR,  'goes_preprocessed.csv')
)
print("✅ Done")
print(f"Size: {os.path.getsize(os.path.join(MY_DIR, 'goes_preprocessed.csv'))/1e6:.1f} MB")

Copying full preprocessed file... (this may take a minute, it's 1.1GB)
✅ Done
Size: 1159.0 MB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

HER_DIR = '/content/drive/MyDrive/BAH_hackathon_data'
MY_DIR  = '/content/drive/MyDrive/isro_hackathon_data'

# Check her file exists first
her_file = os.path.join(HER_DIR, 'goes_preprocessed.csv')
print(f"Her file exists: {os.path.exists(her_file)}")
print(f"Her file size: {os.path.getsize(her_file)/1e6:.1f} MB")

print("\nCopying... (1.1GB, takes 1-2 minutes)")
shutil.copy(her_file, os.path.join(MY_DIR, 'goes_preprocessed.csv'))
print("✅ Done")

# Verify
size = os.path.getsize(os.path.join(MY_DIR, 'goes_preprocessed.csv'))/1e6
print(f"Copied file size: {size:.1f} MB")

Mounted at /content/drive
Her file exists: True
Her file size: 1159.0 MB

Copying... (1.1GB, takes 1-2 minutes)
✅ Done
Copied file size: 1159.0 MB


In [ ]:
import pandas as pd
import os

MY_DIR = '/content/drive/MyDrive/isro_hackathon_data'

goes = pd.read_csv(os.path.join(MY_DIR, 'goes_preprocessed.csv'),
                   usecols=['time'])
goes['time'] = pd.to_datetime(goes['time'], format='mixed')

print(f"Total rows: {len(goes):,}")
print(f"Date range: {goes['time'].min()} to {goes['time'].max()}")
print("\nRows per year:")
print(goes.groupby(goes['time'].dt.year).size())

Total rows: 7,318,564
Date range: 2010-04-07 19:45:00 to 2024-12-31 23:59:00

Rows per year:
time
2010    197139
2011    513282
2012    487995
2013    509454
2014    500472
2015    484422
2016    487872
2017    524874
2018    514760
2019    515372
2020    518987
2021    516145
2022    515490
2023    517304
2024    514996
dtype: int64
